# 01 Data Setup And Train GAN

Use this notebook to verify dataset paths, inspect dataset splits, preview a few real images, and then train a GAN run from the same environment.


In [ ]:
from collections import defaultdict
from pathlib import Path
import json
import os
import random
import sys
import time
from pprint import pprint

import numpy as np
import torch
import torch.nn as nn
from IPython.display import display
from PIL import Image
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, utils as vutils
from torchvision.models import inception_v3

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from config import Config
from models.gan import Generator, ProjectionDiscriminator

ROOT


In [ ]:
def summarize_dataset(data_root, splits=("train", "val", "test")):
    data_root = Path(data_root)
    summary = defaultdict(dict)
    for split in splits:
        split_dir = data_root / split
        if not split_dir.exists():
            continue
        for class_dir in sorted([path for path in split_dir.iterdir() if path.is_dir()]):
            summary[split][class_dir.name] = sum(1 for path in class_dir.iterdir() if path.is_file())
    return {split: dict(class_counts) for split, class_counts in summary.items()}


def preview_training_images(data_root, class_names=None):
    train_root = Path(data_root) / "train"
    if class_names is None:
        class_names = [path.name for path in sorted(train_root.iterdir()) if path.is_dir()]
    for class_name in class_names:
        sample_path = sorted(path for path in (train_root / class_name).glob("*") if path.is_file())[0]
        print(class_name, sample_path.name)
        display(Image.open(sample_path))


In [ ]:
# Edit these values before running.
cfg = Config()
DATA_ROOT = ROOT / "data_final"
FID_ROOT = DATA_ROOT
OUT_DIR = ROOT / "runs" / "gan"

cfg, DATA_ROOT, FID_ROOT, OUT_DIR


In [ ]:
dataset_summary = summarize_dataset(DATA_ROOT)
pprint(dataset_summary)
preview_training_images(DATA_ROOT)


## GAN Training

The cells below keep the notebook-native GAN training workflow intact after the dataset sanity checks above.


In [ ]:
@torch.no_grad()
def get_inception_features(images, model, device, batch_size=64):
    up = torch.nn.Upsample(size=(299, 299), mode="bilinear", align_corners=False)
    mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)
    feats = []
    for i in range(0, len(images), batch_size):
        batch = images[i:i + batch_size].to(device)
        batch = (batch + 1) / 2
        batch = (batch - mean) / std
        batch = up(batch)
        feats.append(model(batch).cpu())
    return torch.cat(feats, dim=0).numpy()


def calc_fid(mu1, sigma1, mu2, sigma2):
    from scipy import linalg
    diff = mu1 - mu2
    covmean = linalg.sqrtm(sigma1 @ sigma2)
    if isinstance(covmean, tuple):
        covmean = covmean[0]
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    return float(diff @ diff + np.trace(sigma1 + sigma2 - 2 * covmean))


def load_inception(device):
    model = inception_v3(weights="IMAGENET1K_V1", transform_input=False)
    model.fc = torch.nn.Identity()
    model.to(device)
    model.eval()
    return model


def make_loader(cfg, data_root, split, train):
    tf_list = [transforms.Resize((cfg.img_size, cfg.img_size))]
    if train:
        tf_list.append(transforms.RandomHorizontalFlip())
    tf_list.extend([
        transforms.ToTensor(),
        transforms.Normalize([0.5] * 3, [0.5] * 3),
    ])
    tf = transforms.Compose(tf_list)
    ds = datasets.ImageFolder(str(Path(data_root) / split), transform=tf)
    loader = DataLoader(ds, batch_size=cfg.gan_batch, shuffle=train, drop_last=train, **cfg.loader_options())
    return loader, ds.classes


def compute_fid(G, real_loader, num_classes, cfg, inception_model, device):
    real_imgs = []
    count = 0
    target = cfg.fid_n_samples
    for imgs, _ in real_loader:
        real_imgs.append(imgs)
        count += imgs.size(0)
        if count >= target:
            break
    real_imgs = torch.cat(real_imgs)[:target]
    n_fid = real_imgs.size(0)

    G.eval()
    fake_imgs = []
    remaining = n_fid
    while remaining > 0:
        bs = min(cfg.gan_batch, remaining)
        z = torch.randn(bs, cfg.z_dim, device=device)
        y = torch.randint(0, num_classes, (bs,), device=device)
        fake_imgs.append(G(z, y).cpu())
        remaining -= bs
    fake_imgs = torch.cat(fake_imgs)[:n_fid]
    G.train()

    real_feats = get_inception_features(real_imgs, inception_model, device)
    fake_feats = get_inception_features(fake_imgs, inception_model, device)
    mu_r, sig_r = real_feats.mean(0), np.cov(real_feats, rowvar=False)
    mu_f, sig_f = fake_feats.mean(0), np.cov(fake_feats, rowvar=False)
    return calc_fid(mu_r, sig_r, mu_f, sig_f)


def save_samples(G, fixed_z, fixed_y, epoch, out_dir, nrow=8):
    out_dir = Path(out_dir)
    G.eval()
    with torch.no_grad():
        imgs = G(fixed_z, fixed_y)
    vutils.save_image(
        imgs,
        out_dir / f"samples_epoch{epoch:04d}.png",
        nrow=nrow,
        normalize=True,
        value_range=(-1, 1),
    )
    G.train()


def train_gan_notebook(cfg, data_root=None, fid_root=None, out_dir=None):
    device = torch.device(cfg.resolve_device())
    print(f"Device: {device}")

    data_root = Path(data_root) if data_root is not None else cfg.data_root
    fid_root = Path(fid_root) if fid_root is not None else data_root
    out_dir = Path(out_dir) if out_dir is not None else Path(cfg.out_root) / "gan"

    torch.manual_seed(cfg.seed)
    random.seed(cfg.seed)
    np.random.seed(cfg.seed)

    train_loader, class_names = make_loader(cfg, data_root=data_root, split="train", train=True)
    num_classes = len(class_names)
    fid_split = cfg.fid_eval_split
    fid_split_path = fid_root / fid_split
    if not fid_split_path.exists():
        fid_split = "train"
    fid_loader, fid_classes = make_loader(cfg, data_root=fid_root, split=fid_split, train=False)
    if fid_classes != class_names:
        raise RuntimeError(f"Class mismatch: {class_names} vs {fid_classes}")

    G = Generator(z_dim=cfg.z_dim, num_classes=num_classes).to(device)
    D = ProjectionDiscriminator(num_classes=num_classes).to(device)
    criterion = nn.BCEWithLogitsLoss()
    opt_G = torch.optim.Adam(G.parameters(), lr=cfg.gan_lr_g, betas=(0.5, 0.999))
    opt_D = torch.optim.Adam(D.parameters(), lr=cfg.gan_lr_d, betas=(0.5, 0.999))

    n_samples = 24
    fixed_z = torch.randn(n_samples, cfg.z_dim, device=device)
    fixed_y = torch.arange(num_classes, device=device).repeat_interleave(n_samples // num_classes)

    out_dir.mkdir(parents=True, exist_ok=True)
    ckpt_dir = out_dir / "checkpoints"
    ckpt_dir.mkdir(exist_ok=True)

    inception_model = load_inception(device)
    fid_log = []
    best_fid = float("inf")
    best_epoch = None
    start_time = time.time()

    for epoch in range(1, cfg.gan_epochs + 1):
        d_loss_acc = g_loss_acc = d_x_acc = d_gz_acc = 0.0
        n_batches = 0
        for real_imgs, real_labels in train_loader:
            real_imgs = real_imgs.to(device)
            real_labels = real_labels.to(device)
            batch_size = real_imgs.size(0)

            real_targets = torch.ones(batch_size, 1, device=device)
            fake_targets = torch.zeros(batch_size, 1, device=device)

            with torch.no_grad():
                z = torch.randn(batch_size, cfg.z_dim, device=device)
                fake_imgs = G(z, real_labels)

            d_real_out = D(real_imgs, real_labels)
            d_fake_out = D(fake_imgs, real_labels)
            d_loss = criterion(d_real_out, real_targets) + criterion(d_fake_out, fake_targets)
            opt_D.zero_grad()
            d_loss.backward()
            nn.utils.clip_grad_norm_(D.parameters(), max_norm=1.0)
            opt_D.step()

            z = torch.randn(batch_size, cfg.z_dim, device=device)
            gen_labels = torch.randint(0, num_classes, (batch_size,), device=device)
            fake_imgs = G(z, gen_labels)
            g_out = D(fake_imgs, gen_labels)
            g_loss = criterion(g_out, torch.ones(batch_size, 1, device=device))
            opt_G.zero_grad()
            g_loss.backward()
            nn.utils.clip_grad_norm_(G.parameters(), max_norm=1.0)
            opt_G.step()

            d_loss_acc += d_loss.item()
            g_loss_acc += g_loss.item()
            d_x_acc += torch.sigmoid(d_real_out).mean().item()
            d_gz_acc += torch.sigmoid(g_out).mean().item()
            n_batches += 1

        d_avg = d_loss_acc / n_batches
        g_avg = g_loss_acc / n_batches
        d_x_avg = d_x_acc / n_batches
        d_gz_avg = d_gz_acc / n_batches

        fid_str = ""
        if epoch % cfg.fid_every == 0 or epoch == cfg.gan_epochs:
            fid = compute_fid(G, fid_loader, num_classes, cfg, inception_model, device)
            fid_log.append({
                "epoch": epoch,
                "fid": fid,
                "d_loss": d_avg,
                "g_loss": g_avg,
                "d_x": d_x_avg,
                "d_gz": d_gz_avg,
            })
            fid_str = f"  FID: {fid:.2f}"
            if fid < best_fid:
                best_fid = fid
                best_epoch = epoch
                torch.save({
                    "epoch": epoch,
                    "fid": fid,
                    "G": G.state_dict(),
                    "D": D.state_dict(),
                    "opt_G": opt_G.state_dict(),
                    "opt_D": opt_D.state_dict(),
                }, ckpt_dir / "best_fid.pt")
        else:
            fid_log.append({
                "epoch": epoch,
                "d_loss": d_avg,
                "g_loss": g_avg,
                "d_x": d_x_avg,
                "d_gz": d_gz_avg,
            })

        print(
            f"[Epoch {epoch:03d}/{cfg.gan_epochs}]  D_loss: {d_avg:.4f}  "
            f"G_loss: {g_avg:.4f}  D(x): {d_x_avg:.3f}  D(G(z)): {d_gz_avg:.3f}{fid_str}"
        )

        if epoch % cfg.sample_every == 0 or epoch == 1:
            save_samples(G, fixed_z, fixed_y, epoch, out_dir)
        if epoch % cfg.ckpt_every == 0 or epoch == cfg.gan_epochs:
            torch.save({
                "epoch": epoch,
                "G": G.state_dict(),
                "D": D.state_dict(),
                "opt_G": opt_G.state_dict(),
                "opt_D": opt_D.state_dict(),
            }, ckpt_dir / f"ckpt_epoch{epoch:04d}.pt")

    with open(out_dir / "train_log.json", "w") as f:
        json.dump(fid_log, f, indent=2)

    summary = {
        "data_root": str(data_root),
        "fid_root": str(fid_root),
        "fid_split": fid_split,
        "num_classes": num_classes,
        "train_samples": len(train_loader.dataset),
        "fid_samples": len(fid_loader.dataset),
        "train_time_sec": round(time.time() - start_time, 1),
        "best_fid": None if best_fid == float("inf") else round(best_fid, 4),
        "best_epoch": best_epoch,
        "out_dir": str(out_dir),
    }
    with open(out_dir / "run_summary.json", "w") as f:
        json.dump(summary, f, indent=2)
    return {"G": G, "D": D, "summary": summary}


In [ ]:
gan_run = train_gan_notebook(cfg, data_root=DATA_ROOT, fid_root=FID_ROOT, out_dir=OUT_DIR)
gan_run["summary"]
